In [1]:
# KCB Loan Portfolio Risk Analyzer
# Phase 1: Data Generation & Ingestion

import pandas as pd
import numpy as np
import sqlite3
import os
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set seed for reproducibility — same "random" data every run
np.random.seed(42)

print("Libraries loaded ✓")

Libraries loaded ✓


In [2]:
# ── SIMULATE KCB RETAIL LOAN PORTFOLIO ──────────────────────────────────────
# 5,000 customer loan records
# Realistic Kenyan banking context: products, regions, currencies

N = 5000

# --- Reference data (realistic KCB context) ---------------------------------

PRODUCTS = ['Personal Loan', 'Mortgage', 'Business Loan',
            'Asset Finance', 'Agricultural Loan', 'Staff Loan']

PRODUCT_WEIGHTS = [0.35, 0.15, 0.25, 0.10, 0.10, 0.05]

BRANCHES = [
    'Nairobi CBD', 'Westlands', 'Mombasa', 'Kisumu',
    'Nakuru', 'Eldoret', 'Thika', 'Machakos',
    'Nyeri', 'Meru', 'Garissa', 'Kisii'
]

LOAN_STATUS = ['Performing', 'Watch', 'Substandard', 'Doubtful', 'Loss']
# CBK loan classification — this is the real Kenyan regulatory framework

STATUS_WEIGHTS = [0.72, 0.12, 0.07, 0.05, 0.04]
# 28% NPL rate — slightly above CBK sector average, realistic for simulation

RELATIONSHIP_MANAGERS = [
    'J. Kamau', 'A. Odhiambo', 'F. Wanjiku', 'P. Mutua',
    'S. Kipchoge', 'M. Auma', 'D. Njoroge', 'R. Mwangi',
    'C. Otieno', 'B. Chebet'
]

# Loan officer notes — unstructured text (this is our NLP goldmine later)
NOTE_TEMPLATES = [
    "Customer has maintained regular payments. Business performing well in {}.",
    "Missed {} instalments. Customer cited {} as reason for default.",
    "Loan restructured in {}. Customer agreed to revised repayment schedule.",
    "Site visit conducted. Business operational but cash flow strained due to {}.",
    "Customer unresponsive. {} attempts to contact via phone failed.",
    "Strong repayment history. Recommend for limit increase.",
    "Collateral verified: {}. Market value adequate.",
    "Customer requested deferral citing {}. Under review.",
    "Account flagged for review. Irregular deposits noted.",
    "New customer. KYC complete. First disbursement processed."
]

BUSINESSES = ['retail shop', 'matatu business', 'farm produce', 'salon',
              'hardware store', 'restaurant', 'real estate', 'logistics']
REASONS = ['illness', 'school fees', 'drought', 'business losses',
           'retrenchment', 'family emergency', 'COVID-19 effects']
COLLATERAL = ['title deed', 'motor vehicle logbook', 'shares certificate',
              'fixed deposit', 'land parcel LR No.']

def generate_loan_note(status):
    """Generate realistic loan officer notes based on loan status."""
    if status == 'Performing':
        templates = [NOTE_TEMPLATES[0], NOTE_TEMPLATES[5], NOTE_TEMPLATES[9]]
        note = np.random.choice(templates)
        return note.format(np.random.choice(BUSINESSES))
    elif status in ['Watch', 'Substandard']:
        templates = [NOTE_TEMPLATES[1], NOTE_TEMPLATES[3], NOTE_TEMPLATES[7]]
        note = np.random.choice(templates)
        try:
            return note.format(np.random.randint(1, 4), np.random.choice(REASONS))
        except:
            return note.format(np.random.choice(REASONS))
    else:  # Doubtful or Loss
        templates = [NOTE_TEMPLATES[2], NOTE_TEMPLATES[4], NOTE_TEMPLATES[8]]
        note = np.random.choice(templates)
        try:
            return note.format(np.random.randint(3, 8))
        except:
            return note

# --- Generate core loan data -------------------------------------------------

products = np.random.choice(PRODUCTS, size=N, p=PRODUCT_WEIGHTS)
statuses = np.random.choice(LOAN_STATUS, size=N, p=STATUS_WEIGHTS)

# Loan amounts vary by product (in KES)
amount_ranges = {
    'Personal Loan':     (50_000,   500_000),
    'Mortgage':          (2_000_000, 15_000_000),
    'Business Loan':     (100_000,  3_000_000),
    'Asset Finance':     (500_000,  5_000_000),
    'Agricultural Loan': (30_000,   500_000),
    'Staff Loan':        (50_000,   1_000_000),
}

loan_amounts = np.array([
    np.random.randint(*amount_ranges[p]) for p in products
])

# Disbursement dates over last 5 years
start_date = datetime(2020, 1, 1)
end_date   = datetime(2024, 12, 31)
date_range = (end_date - start_date).days

disbursement_dates = [
    start_date + timedelta(days=int(np.random.randint(0, date_range)))
    for _ in range(N)
]

# Loan tenor in months by product
tenor_map = {
    'Personal Loan': (12, 60),
    'Mortgage': (60, 240),
    'Business Loan': (12, 84),
    'Asset Finance': (24, 60),
    'Agricultural Loan': (6, 24),
    'Staff Loan': (12, 48),
}
tenors = np.array([
    np.random.randint(*tenor_map[p]) for p in products
])

# Interest rates (%) — realistic Kenyan market rates
interest_rates = np.round(np.random.uniform(12.5, 18.5, N), 2)

df_loans = pd.DataFrame({
    'loan_id':              [f'KCB{str(i).zfill(6)}' for i in range(1, N+1)],
    'customer_id':          [f'CUS{str(np.random.randint(1, 4000)).zfill(5)}' for _ in range(N)],
    'product':              products,
    'branch':               np.random.choice(BRANCHES, size=N),
    'relationship_manager': np.random.choice(RELATIONSHIP_MANAGERS, size=N),
    'disbursement_date':    disbursement_dates,
    'loan_amount_kes':      loan_amounts,
    'tenor_months':         tenors,
    'interest_rate_pct':    interest_rates,
    'loan_status':          statuses,
    'loan_officer_notes':   [generate_loan_note(s) for s in statuses],
})

print(f"Generated {len(df_loans):,} loan records ✓")
df_loans.head(3)

Generated 5,000 loan records ✓


,loan_id,customer_id,product,branch,relationship_manager,disbursement_date,loan_amount_kes,tenor_months,interest_rate_pct,loan_status,loan_officer_notes
0,KCB000001,CUS03679,Mortgage,Kisii,J. Kamau,2023-10-30,12939574,89,13.91,Performing,New customer. KYC complete. First disbursement...
1,KCB000002,CUS00782,Staff Loan,Machakos,R. Mwangi,2021-09-23,687475,12,17.87,Performing,Customer has maintained regular payments. Busi...
2,KCB000003,CUS01910,Business Loan,Mombasa,M. Auma,2021-08-14,996881,40,12.77,Substandard,Customer requested deferral citing 1. Under re...


In [3]:
# ── INJECT REALISTIC DATA QUALITY ISSUES ────────────────────────────────────
# Real banking data is never clean. We add the exact types of mess
# you will encounter in a real core banking system export.

df_messy = df_loans.copy()

# 1. Missing values — loan officer notes sometimes not captured
mask_missing_notes = np.random.random(N) < 0.08  # 8% missing
df_messy.loc[mask_missing_notes, 'loan_officer_notes'] = np.nan

# 2. Missing branch data — system migration artefact
mask_missing_branch = np.random.random(N) < 0.04
df_messy.loc[mask_missing_branch, 'branch'] = np.nan

# 3. Inconsistent product naming — manual data entry errors
product_typos = {
    'Personal Loan':     ['personal loan', 'Personal  Loan', 'PERSONAL LOAN', 'Pers. Loan'],
    'Business Loan':     ['business loan', 'Biz Loan', 'BUSINESS LOAN', 'Business  Loan'],
    'Agricultural Loan': ['Agri Loan', 'agricultural loan', 'Agric Loan', 'AGRI'],
}
for correct, variants in product_typos.items():
    mask = (df_messy['product'] == correct) & (np.random.random(N) < 0.15)
    df_messy.loc[mask, 'product'] = np.random.choice(variants, mask.sum())

# 4. Outlier loan amounts — data entry errors (extra zeros)
mask_outlier = np.random.random(N) < 0.01
df_messy.loc[mask_outlier, 'loan_amount_kes'] *= 10

# 5. Future disbursement dates — system errors
mask_future = np.random.random(N) < 0.005
df_messy.loc[mask_future, 'disbursement_date'] = datetime(2026, 6, 1)

# 6. Duplicate records — common in CBS exports
duplicates = df_messy.sample(25)
df_messy = pd.concat([df_messy, duplicates], ignore_index=True)

print(f"Messy dataset shape: {df_messy.shape}")
print(f"\nNull counts:")
print(df_messy.isnull().sum()[df_messy.isnull().sum() > 0])
print(f"\nDuplicate rows: {df_messy.duplicated(subset='loan_id').sum()}")

Messy dataset shape: (5025, 11)

Null counts:
branch                220
loan_officer_notes    380
dtype: int64

Duplicate rows: 25


In [4]:
#  DATA CLEANING
# Document every decision — you must be able to explain each one

df_clean = df_messy.copy()

# 1. Remove duplicates — keep first occurrence
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset='loan_id', keep='first')
print(f"Duplicates removed: {before - len(df_clean)}")

# 2. Standardise product names — map all variants to canonical names
product_map = {
    'personal loan':    'Personal Loan',
    'personal  loan':   'Personal Loan',
    'pers. loan':       'Personal Loan',
    'business loan':    'Business Loan',
    'biz loan':         'Business Loan',
    'business  loan':   'Business Loan',
    'agri loan':        'Agricultural Loan',
    'agric loan':       'Agricultural Loan',
    'agricultural loan':'Agricultural Loan',
    'agri':             'Agricultural Loan',
}
df_clean['product'] = (
    df_clean['product']
    .str.strip()
    .str.lower()
    .replace(product_map)
    .str.title()
)

# 3. Cap outlier loan amounts — flag don't delete
df_clean['amount_outlier_flag'] = False
product_ceilings = {
    'Personal Loan':     500_000,
    'Mortgage':          15_000_000,
    'Business Loan':     3_000_000,
    'Asset Finance':     5_000_000,
    'Agricultural Loan': 500_000,
    'Staff Loan':        1_000_000,
}
for product, ceiling in product_ceilings.items():
    mask = (df_clean['product'] == product) & (df_clean['loan_amount_kes'] > ceiling * 1.5)
    df_clean.loc[mask, 'amount_outlier_flag'] = True

print(f"Outlier amounts flagged: {df_clean['amount_outlier_flag'].sum()}")

# 4. Remove future disbursement dates — replace with NaN, investigate later
today = pd.Timestamp.today()
df_clean['disbursement_date'] = pd.to_datetime(df_clean['disbursement_date'])
mask_future = df_clean['disbursement_date'] > today
df_clean.loc[mask_future, 'disbursement_date'] = pd.NaT
print(f"Future dates nulled: {mask_future.sum()}")

# 5. Fill missing branch with 'Unknown' — preserves row, flags for follow-up
df_clean['branch'] = df_clean['branch'].fillna('Unknown')

# 6. Derive useful columns
df_clean['year_disbursed'] = df_clean['disbursement_date'].dt.year
df_clean['month_disbursed'] = df_clean['disbursement_date'].dt.month
df_clean['is_npl'] = df_clean['loan_status'].isin(
    ['Substandard', 'Doubtful', 'Loss']
).astype(int)
df_clean['has_notes'] = df_clean['loan_officer_notes'].notna().astype(int)

print(f"\nFinal cleaned shape: {df_clean.shape}")
print(f"NPL loans: {df_clean['is_npl'].sum():,} ({df_clean['is_npl'].mean()*100:.1f}%)")

Duplicates removed: 25
Outlier amounts flagged: 57
Future dates nulled: 29

Final cleaned shape: (5000, 16)
NPL loans: 763 (15.3%)


In [5]:
#  CBK SECTOR BENCHMARK DATA ─────────────
# Source: Central Bank of Kenya — Bank Supervision Annual Reports
# https://www.centralbank.go.ke/bank-supervision/
# Real published figures, manually structured here for reliability

cbk_data = {
    'year': [2020, 2021, 2022, 2023, 2024],

    # Gross NPL ratio for Kenyan banking sector (%)
    # Source: CBK Bank Supervision Annual Reports
    'sector_npl_ratio_pct': [14.1, 14.0, 14.8, 15.7, 16.4],

    # Private sector credit growth (%)
    'credit_growth_pct': [7.7, 8.6, 12.5, 13.9, 11.2],

    # Average lending rate (%)
    'avg_lending_rate_pct': [12.03, 12.09, 12.45, 14.26, 16.08],

    # Average deposit rate (%)
    'avg_deposit_rate_pct': [6.05, 6.15, 6.84, 8.42, 9.17],

    # Total banking sector loans (KES Billions)
    'sector_loans_kes_bn': [3_142, 3_426, 3_856, 4_290, 4_614],

    # Number of commercial banks
    'num_commercial_banks': [38, 38, 38, 38, 38],
}

df_cbk = pd.DataFrame(cbk_data)

print("CBK macro dataset:")
print(df_cbk)

CBK macro dataset:
   year  sector_npl_ratio_pct  credit_growth_pct  avg_lending_rate_pct  \
0  2020                  14.1                7.7                 12.03   
1  2021                  14.0                8.6                 12.09   
2  2022                  14.8               12.5                 12.45   
3  2023                  15.7               13.9                 14.26   
4  2024                  16.4               11.2                 16.08   

   avg_deposit_rate_pct  sector_loans_kes_bn  num_commercial_banks  
0                  6.05                 3142                    38  
1                  6.15                 3426                    38  
2                  6.84                 3856                    38  
3                  8.42                 4290                    38  
4                  9.17                 4614                    38  


In [6]:
# ── LOAD TO SQL DATABASE ─────────────────────────────────────────────────────

DB_PATH = 'kcb_portfolio.db'
conn = sqlite3.connect(DB_PATH)

# Table 1: loan portfolio
df_clean.to_sql('loans', conn, if_exists='replace', index=False)

# Table 2: CBK macro benchmarks
df_cbk.to_sql('cbk_benchmarks', conn, if_exists='replace', index=False)

# Verify
cursor = conn.cursor()
for table in ['loans', 'cbk_benchmarks']:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    print(f"{table}: {cursor.fetchone()[0]:,} rows ✓")

conn.close()

loans: 5,000 rows ✓
cbk_benchmarks: 5 rows ✓


In [7]:
#  VERIFICATION QUERIES 

conn = sqlite3.connect(DB_PATH)

query = """
    SELECT
        l.year_disbursed                            AS year,
        COUNT(*)                                    AS total_loans,
        ROUND(SUM(l.loan_amount_kes) / 1e6, 1)     AS portfolio_kes_millions,
        ROUND(AVG(l.interest_rate_pct), 2)          AS avg_interest_rate,
        ROUND(SUM(l.is_npl) * 100.0 / COUNT(*), 1) AS portfolio_npl_pct,
        c.sector_npl_ratio_pct                      AS cbk_sector_npl_pct
    FROM loans l
    LEFT JOIN cbk_benchmarks c ON l.year_disbursed = c.year
    WHERE l.year_disbursed IS NOT NULL
    GROUP BY l.year_disbursed, c.sector_npl_ratio_pct
    ORDER BY l.year_disbursed
"""

df_verify = pd.read_sql_query(query, conn)
conn.close()

print("=== PORTFOLIO vs CBK BENCHMARK ===")
df_verify

=== PORTFOLIO vs CBK BENCHMARK ===


,year,total_loans,portfolio_kes_millions,avg_interest_rate,portfolio_npl_pct,cbk_sector_npl_pct
0,2020.0,970,2340.7,15.53,15.6,14.1
1,2021.0,994,2142.9,15.52,15.7,14.0
2,2022.0,969,2528.4,15.51,14.0,14.8
3,2023.0,1004,2312.6,15.68,16.7,15.7
4,2024.0,1034,2231.0,15.53,14.0,16.4
